# HeteroGAT retraining on Google Colab (T4 GPU)

Retrains the HeteroGAT with device/IP binding (PNA) with the **fixed loader** (isolated
per-target subgraphs). Settings: `configs/hetero_gat_colab.yaml`.

**Before you start**
1. Runtime → Change runtime type → **T4 GPU**.
2. Push the current code to GitHub (this notebook clones it).
3. Upload the data files listed in `notebooks/colab_manifest.json` to Google Drive under
   `MyDrive/fraudgnn/` with the same relative paths:
   `data/processed/train_graph.pt`, `val_graph.pt`, `test_graph.pt`, `data/raw/transactions.parquet`.

**Where things go**
- Resume state (model, optimizer, scheduler, epoch, best val PR-AUC, patience, RNG) is
  written to Drive after every epoch: `MyDrive/fraudgnn/checkpoints/`.
- MLflow database, best checkpoint and logs stay in the local run folder `/content/run`
  and are copied to `MyDrive/fraudgnn/runs/` by the last cell.

**If the session disconnects:** reconnect and run all cells again. Training continues from
the last finished epoch (`--resume`). If the old MLflow run is gone with the VM, a new run is
started, tagged `resumed_from_run`, and the per-epoch history is replayed into it.

In [ ]:
# Settings (must match the paths in configs/hetero_gat_colab.yaml)
REPO_URL = "https://github.com/CodeMaann/Real-Time-Fraud-Detection-with-Graph-Neural-Networks.git"
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/fraudgnn"
CONFIG = "configs/hetero_gat_colab.yaml"
REPO_DIR = "/content/repo"
LOCAL_ROOT = "/content"          # data is copied to /content/data/...
RUN_DIR = "/content/run"         # MLflow DB, best checkpoint, logs
RUN_NAME = "hetero_gat_pna_isolated"

## 1. Mount Google Drive and check the GPU

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Clone the repository

In [ ]:
!rm -rf {REPO_DIR}
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} log --oneline -1

## 3. Install torch-geometric and the project

Colab already provides PyTorch with CUDA. The graph files are pickled PyG objects, so
torch-geometric is pinned to the version that wrote them (see the manifest).

In [ ]:
import json

manifest = json.load(open(f"{REPO_DIR}/notebooks/colab_manifest.json"))
PYG_VERSION = manifest["created_with"]["torch_geometric"]
!pip install -q torch-geometric=={PYG_VERSION} mlflow scikit-learn
!pip install -q -e {REPO_DIR} --no-deps

import torch, torch_geometric
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| available", torch.cuda.is_available())
print("torch_geometric", torch_geometric.__version__, "| manifest", manifest["created_with"])
assert torch.cuda.is_available(), "No GPU: set Runtime -> Change runtime type -> T4 GPU"

## 4. Copy the data from Drive to local disk and verify checksums

In [ ]:
import hashlib, shutil
from pathlib import Path

def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

bad = []
for entry in manifest["files"]:
    src = Path(DRIVE_ROOT) / entry["path"]
    dst = Path(LOCAL_ROOT) / entry["path"]
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists() or dst.stat().st_size != entry["bytes"]:
        shutil.copy2(src, dst)
    ok = sha256(dst) == entry["sha256"]
    print(f"{'OK ' if ok else 'BAD'} {entry['path']:<36} {entry['mb']:>7.1f} MB")
    if not ok:
        bad.append(entry["path"])
assert not bad, f"Checksum mismatch: {bad}. Re-upload these files."

## 5. Check that the config paths match this notebook and the graphs load

In [ ]:
import os, sys
os.chdir(REPO_DIR)
from fraudgnn.config import load_gnn_config

cfg = load_gnn_config(CONFIG)
assert Path(cfg.data_dir).as_posix() == f"{LOCAL_ROOT}/data/processed", cfg.data_dir
assert Path(cfg.training.best_model_dir).as_posix().startswith(RUN_DIR), cfg.training.best_model_dir
assert Path(cfg.training.checkpoint_path).as_posix().startswith(DRIVE_ROOT), cfg.training.checkpoint_path
Path(RUN_DIR).mkdir(parents=True, exist_ok=True)
Path(cfg.training.checkpoint_path).parent.mkdir(parents=True, exist_ok=True)

for name in ("train", "val", "test"):
    g = torch.load(f"{cfg.data_dir}/{name}_graph.pt", weights_only=False)
    print(name, {nt: g[nt].num_nodes for nt in g.node_types})
print("resume state:", cfg.training.checkpoint_path,
      "(exists)" if Path(cfg.training.checkpoint_path).exists() else "(none yet)")

## 6. GPU timing (run this first)

Times 20 optimizer steps (each `batch_size` targets, in `micro_batch_size` chunks) and 20
validation batches, then prints seconds per epoch, the estimate for the configured maximum
epochs, and peak GPU memory. Nothing is saved. If peak memory is far below the T4's 15 GB,
you can raise `micro_batch_size` to 1024 in the config (same gradient, fewer passes).

In [ ]:
!cd {REPO_DIR} && python -u -m fraudgnn.training.train_gnn --config {CONFIG} --causal --device cuda --time-batches 20 2>&1 | tee {RUN_DIR}/timing.log

## 7. Train (resumable)

Writes the resume state to Drive after every epoch. Re-running this cell after a disconnect
continues from the last finished epoch. The log is appended to `{RUN_DIR}/train.log`.
To split the run across sessions, add `--stop-after-epochs N`.

In [ ]:
!cd {REPO_DIR} && python -u -m fraudgnn.training.train_gnn --config {CONFIG} --causal --device cuda --resume 2>&1 | tee -a {RUN_DIR}/train.log

## 8. Copy the best checkpoint, MLflow database and logs back to Drive

In [ ]:
from datetime import datetime

dest = Path(DRIVE_ROOT) / "runs" / f"{RUN_NAME}_{datetime.now():%Y%m%d_%H%M%S}"
shutil.copytree(RUN_DIR, dest, dirs_exist_ok=True)
for p in sorted(dest.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(dest)}  {p.stat().st_size / 1e6:.1f} MB")
print("copied to", dest)